# 04 · Transformer Encoder

Linear projection + sinusoidal positions → 3 pre-norm encoder layers → last-token head.

**Protocol (identical for all three models — see `lob/`):**
- Chronological split 70 / 15 / 15 (train / val / test), with purging so that no label window crosses a split boundary.
- Early stopping on **validation** loss; the best-validation weights are restored.
- The per-class probability threshold is chosen on **validation** Sharpe and applied unchanged to test.
- Backtest on non-overlapping 10-second steps, fee 1.5 bps × |Δposition|.

In [ ]:
# ── Setup: works on Colab (clones the repo) and locally (run from notebooks/) ──
import os, sys

REPO_URL = "https://github.com/<your-username>/mat629-lob-prediction.git"   # TODO: your repo

if "google.colab" in sys.modules:
    if not os.path.exists("mat629-lob-prediction"):
        !git clone -q $REPO_URL
    %cd mat629-lob-prediction
    !pip install -q kagglehub pyarrow
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

# Where the feature table lives. On Colab you can point this at Google Drive
# (after drive.mount) so it survives runtime restarts.
FEATURES_PATH = "data/features.parquet"

In [ ]:
import torch
from lob.data import FEATURE_SETS, load_dataset
from lob.experiment import run_experiment, save_summary
from lob.features import build_dataset

if not os.path.exists(FEATURES_PATH):
    build_dataset(out_path=FEATURES_PATH)          # ~1M rows; downloads via kagglehub
df, meta = load_dataset(FEATURES_PATH)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| rows:", len(df))

## Hyperparameters

In [ ]:
SEQ_LEN, BATCH_SIZE = 30, 512
MODEL_KWARGS = dict(d_model=64, n_heads=4, n_layers=3, d_ff=128, dropout=0.1)
TRAIN_KWARGS = dict(epochs=30, lr=1e-3, weight_decay=1e-4, patience=6,
                    scheduler="onecycle", monitor="loss")

## Run all feature sets

In [ ]:
rows, models = [], {}
for fs in FEATURE_SETS:
    summary, models[fs], _ = run_experiment(df, meta, "transformer", fs, MODEL_KWARGS, TRAIN_KWARGS, SEQ_LEN, BATCH_SIZE, device=DEVICE)
    rows.append(summary)

## Summary

In [ ]:
summary = save_summary(rows, "transformer")
summary.style.format({c: "{:.2%}" for c in summary.columns if c.endswith(("return", "mdd"))}
                     | {c: "{:.4f}" for c in ["test_acc", "test_dir_f1"]}
                     | {c: "{:.2f}" for c in summary.columns if c.endswith("sharpe")})